# Mushroom Classification Using Machine Learning

A reproducible comparison of three supervised classification algorithms for predicting mushroom edibility from the UCI Mushroom dataset:

- Decision Tree
- Random Forest
- Support Vector Machine (RBF kernel)

This notebook preserves the original project's three-model scope while improving preprocessing, evaluation, reproducibility, and visual reporting.

> **Safety note:** This is an educational dataset experiment. Never use this model to identify or decide whether real wild mushrooms are safe to eat.

## 1. Imports and reproducibility

The preprocessing steps are placed inside scikit-learn pipelines. This ensures that encoders are fitted only on each training fold during cross-validation and avoids preprocessing leakage.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

# Resolve the dataset path whether this notebook is run from the repository root
# or from the notebooks/ directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data" / "agaricus-lepiota.data"
IMAGE_DIR = REPO_ROOT / "images"
IMAGE_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Load and inspect the dataset

In [ ]:
COLUMNS = [
    "class", "cap-shape", "cap-surface", "cap-color", "bruises",
    "odor", "gill-attachment", "gill-spacing", "gill-size",
    "gill-color", "stalk-shape", "stalk-root",
    "stalk-surface-above-ring", "stalk-surface-below-ring",
    "stalk-color-above-ring", "stalk-color-below-ring",
    "veil-type", "veil-color", "ring-number", "ring-type",
    "spore-print-color", "population", "habitat",
]

df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS)

print(f"Dataset shape: {df.shape}")
display(df.head())
display(df["class"].value_counts().rename(index={"e": "edible", "p": "poisonous"}).to_frame("count"))

# The source dataset uses '?' to represent an unknown stalk-root category.
unknown_counts = (df == "?").sum().sort_values(ascending=False)
display(unknown_counts[unknown_counts > 0].to_frame("unknown_values"))

### Dataset notes

The dataset contains 8,124 observations, 22 input attributes, and a binary target (`e` = edible, `p` = poisonous). The `?` values in `stalk-root` are retained as a categorical value rather than silently discarded. One-hot encoding can represent this category, and `handle_unknown="ignore"` allows the pipeline to handle categories not seen during fitting.

## 3. Prepare features and target

In [ ]:
X = df.drop(columns="class")
y = df["class"].map({"e": "edible", "p": "poisonous"})

if y.isna().any():
    raise ValueError("Unexpected target labels found in the dataset.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")
print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

## 4. Define preprocessing and models

All predictors are categorical. `OneHotEncoder` is used instead of assigning arbitrary integer codes to categories. Decision Tree and Random Forest use the encoded features directly. SVM additionally uses `StandardScaler(with_mean=False)` after one-hot encoding.

In [ ]:
categorical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            list(X.columns),
        )
    ],
    remainder="drop",
)

models = {
    "Decision Tree": Pipeline(
        steps=[
            ("preprocessor", categorical_preprocessor),
            ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", categorical_preprocessor),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=100,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "Support Vector Machine": Pipeline(
        steps=[
            ("preprocessor", categorical_preprocessor),
            ("scaler", StandardScaler(with_mean=False)),
            ("classifier", SVC(kernel="rbf")),
        ]
    ),
}

## 5. Train and evaluate the three models

In [ ]:
fitted_models = {}
predictions = {}
metric_rows = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    fitted_models[model_name] = model
    predictions[model_name] = y_pred

    metric_rows.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision (poisonous)": precision_score(
                y_test, y_pred, pos_label="poisonous", zero_division=0
            ),
            "Recall (poisonous)": recall_score(
                y_test, y_pred, pos_label="poisonous", zero_division=0
            ),
            "F1-score (poisonous)": f1_score(
                y_test, y_pred, pos_label="poisonous", zero_division=0
            ),
        }
    )

    print(f"\n{model_name}")
    print(classification_report(y_test, y_pred, labels=["edible", "poisonous"], zero_division=0))

metrics_df = pd.DataFrame(metric_rows).set_index("Model")
display(metrics_df.style.format("{:.4f}"))

### Interpreting the metrics

Accuracy is the proportion of all test observations classified correctly. Precision for the poisonous class measures how often predictions of poisonous are correct; recall measures how many poisonous observations are detected. The F1-score combines precision and recall.

For a safety-related classification problem, accuracy alone is insufficient. In real-world settings, false negatives and the consequences of incorrect predictions would require careful domain-specific analysis.

## 6. Confusion matrices

In [ ]:
for model_name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=["edible", "poisonous"])
    display(pd.DataFrame(
        cm,
        index=["Actual edible", "Actual poisonous"],
        columns=["Predicted edible", "Predicted poisonous"],
    ).rename_axis(model_name))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["edible", "poisonous"],
    ).plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"{model_name} — Confusion Matrix")
    fig.tight_layout()
    safe_name = model_name.lower().replace(" ", "_")
    fig.savefig(IMAGE_DIR / f"{safe_name}_confusion_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()

## 7. Compare model performance

In [ ]:
plot_metrics = metrics_df[["Accuracy", "Precision (poisonous)", "Recall (poisonous)", "F1-score (poisonous)"]]
ax = plot_metrics.plot(kind="bar", figsize=(10, 5), ylim=(0, 1.05))
ax.set_title("Test-set performance comparison")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
ax.tick_params(axis="x", rotation=15)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "model_performance_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Five-fold stratified cross-validation

Cross-validation estimates performance across multiple train/validation splits. The preprocessing pipeline is refitted within each fold, so the encoder does not learn categories from the validation fold.

In [ ]:
cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_rows = []
for model_name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=1,
    )
    cv_rows.append(
        {
            "Model": model_name,
            "Mean CV accuracy": scores.mean(),
            "Standard deviation": scores.std(),
            "Fold scores": np.round(scores, 5).tolist(),
        }
    )

cv_results = pd.DataFrame(cv_rows).set_index("Model")
display(cv_results)

## 9. Random Forest feature importance

In [ ]:
rf_pipeline = fitted_models["Random Forest"]
rf_classifier = rf_pipeline.named_steps["classifier"]
encoded_feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()

feature_importance = (
    pd.DataFrame(
        {
            "Encoded feature": encoded_feature_names,
            "Importance": rf_classifier.feature_importances_,
        }
    )
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(15))

top_features = feature_importance.head(15).sort_values("Importance")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_features["Encoded feature"], top_features["Importance"])
ax.set_title("Top 15 Random Forest encoded-feature importances")
ax.set_xlabel("Feature importance")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "random_forest_feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Results and limitations

Use the metric tables and figures produced above as the results for this run. Results are dependent on the dataset, random seed, preprocessing, and software versions.

The original dataset is a well-known, structured benchmark. Very high scores on this dataset do not establish that a model can safely identify wild mushrooms in the field. The dataset may not represent the full range of species, environments, image quality, or uncertainty encountered outside the benchmark.

This project is an educational comparison of three classification algorithms, not a food-safety or foraging application.

## 11. Reproducibility

From the repository root, install the dependencies and launch Jupyter:

```bash
python -m venv .venv
# Windows:
.venv\\Scripts\\activate
# macOS / Linux:
source .venv/bin/activate

pip install -r requirements.txt
jupyter notebook notebooks/mushroom_classification.ipynb
```

The notebook uses a fixed random seed (`42`) for the train/test split, Decision Tree, Random Forest, and cross-validation. SVM uses the same split and preprocessing pipeline.